### Задание 1. 

In [74]:
import numpy as np
import scipy.stats as stats
import pandas as pd
import matplotlib.pyplot as plt

In [75]:
df = pd.read_csv('mobile_phones.csv')
df.head()

,battery_power,blue,clock_speed,dual_sim,fc,four_g,int_memory,m_dep,mobile_wt,n_cores,...,px_height,px_width,ram,sc_h,sc_w,talk_time,three_g,touch_screen,wifi,price_range
0,842,0,2.2,0,1,0,7,0.6,188,2,...,20,756,2549,9,7,19,0,0,1,1
1,1021,1,0.5,1,0,1,53,0.7,136,3,...,905,1988,2631,17,3,7,1,1,0,2
2,563,1,0.5,1,2,1,41,0.9,145,5,...,1263,1716,2603,11,2,9,1,1,0,2
3,615,1,2.5,0,0,0,10,0.8,131,6,...,1216,1786,2769,16,8,11,1,0,0,2
4,1821,1,1.2,0,13,1,44,0.6,141,2,...,1208,1212,1411,8,2,15,1,1,0,1


In [76]:
data = df[['sc_h', 'sc_w', 'battery_power', 'mobile_wt']]
print("Пропущенных значений:\n", data.isna().sum())

Пропущенных значений:
 sc_h             0
sc_w             0
battery_power    0
mobile_wt        0
dtype: int64


Мы проверили, что все данные корректны. Теперь можно составить матрицу размером n * 4, где n - количество строк с данными, а 4 - количество признаков + 1 (+1  нужен для того, чтобы сделать столбец из единиц, которые впоследствии будут домножаться на число, образовывая свободный коэффициент)

In [77]:
X_without_1 = data.drop('mobile_wt', axis=1).values
y = data['mobile_wt'].values
n = y.shape[0]
X = np.column_stack([np.ones(n), X_without_1])
p = 4 # количество параметров, включая свободный коэффициент

Рассчёт оценок коэффициентов модели МНК. Коэффициенты находятся по формуле:

$
\beta = (X^TX)^{-1}X^Ty
$


In [78]:
first = np.dot(X.T, X)
inv_first = np.linalg.inv(first)
beta = np.dot(inv_first, np.dot(X.T, y))

cols = data.columns[:3].tolist()
cols = ['свободный коэффициент'] + cols

for i in range(4):
    print(cols[i], ": ", beta[i])

свободный коэффициент :  143.64057226048226
sc_h :  -0.26354816931240066
sc_w :  -0.039548615944461574
battery_power :  6.44803879089606e-05


In [79]:
y_pred = np.dot(X, beta)
diff = y - y_pred
rss = (diff**2).sum()

In [80]:
rss

np.float64(2502101.030648562)

Оценим остаточную дисперсию

In [81]:
sigma2 = rss / (n - p)

In [82]:
sigma2

np.float64(1253.557630585452)

Теперь нужно построить ковариационную матрицу:
$
Cov(\beta) = \sigma^2 \cdot (X^TX)^{-1}
$

И вычислить стандартные ошибки оценок коэффициентов (их квадраты находятся на главной диагонали ковариационной матрицы).

In [83]:
var_beta = sigma2 * inv_first
se_beta = np.sqrt(np.diag(var_beta))

Теперь мы уже сможем построить довериательные интервалы, зная стандатную ошибку se_beta. Для этого мы ищем квантили распределения Стьюдента.

In [98]:
t = stats.t.ppf(0.975, df=n-p)
interval = np.column_stack([beta - t * se_beta, beta + t * se_beta])

for i in range(4):
    print(cols[i], ": ", interval[i])

свободный коэффициент :  [137.05003113 150.23111339]
sc_h :  [-0.69104605  0.16394972]
sc_w :  [-0.45290805  0.37381082]
battery_power :  [-0.00347147  0.00360043]


In [85]:
tss = ((y - y.mean())**2).sum()
r2 = 1 - (rss / tss)
r2

np.float64(0.0011644496581527664)

##### Гипотеза 1. Чем больше высота экрана, тем больше масса.

$$
H_0: \beta_1 \leq 0 \\

H_1: \beta_1 > 0
$$

Где $\beta_1$ - коэффициент при признаке "высота экрана". 

In [88]:
t_stat = beta[1] / se_beta[1]
p_value_h = 1 - stats.t.cdf(t_stat, df=n-p)

p_value_h

np.float64(0.8866029986025021)

##### Гипотеза 2. Чем больше ширина экрана, тем больше масса

$$
H_0: \beta_2 \leq 0 \\

H_1: \beta_2 > 0
$$

Где $\beta_2$ - коэффициент при признаке "ширина экрана". 

In [90]:
t_stat = beta[2] / se_beta[2]
p_value_w = 1 - stats.t.cdf(t_stat, df=n-p)

p_value_w

np.float64(0.5744092575407408)

##### Гипотеза 3. Коэффициенты при ширине экрана и емкости аккумулятора оба равны нулю.

$$
H_0: \beta_2 = 0 \quad and \quad \beta_3 = 0\\

H_1: \beta_2 \neq 0 \quad or \quad \beta_3 \neq 0
$$

In [97]:
R = np.array([[0, 0, 1, 0], [0, 0, 0, 1]])
r_vector = np.array([0, 0])

R_beta = np.dot(R, beta)
diff_R = R_beta - r_vector

scaled = np.dot(np.dot(R, inv_first), R.T) / sigma2
middle = np.linalg.inv(scaled)

F_stat = (np.dot(diff_R.T, np.dot(middle, diff_R))) / 2
p_value_F = 1 - stats.f.cdf(F_stat, dfn=2, dfd=n-p)

format(p_value_F, '.10f')

'0.0000000000'